In [1]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torchvision import models, transforms
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image

warnings.filterwarnings("ignore")
tqdm.pandas()

In [2]:
DEVICE = "mps" if torch.mps.is_available() else "cpu"

DATA_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/competition")
IMAGE_FOLDER = Path(r"/Users/semyonkuricyn/vs/RWB_ML/ML_solve/Semøn/data/images")

print(DEVICE)

mps


In [3]:
res_net50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(DEVICE)

In [4]:
res_net18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE)

In [6]:
res_net34 = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1).to(DEVICE)

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /Users/semyonkuricyn/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:16<00:00, 5.37MB/s]


In [7]:
train_df = pd.read_csv(DATA_FOLDER / "train.csv")
test_df = pd.read_csv(DATA_FOLDER / "test.csv")

# val_df = train_df.head(500)

In [8]:
tr = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

In [9]:
class ImbaDataset(Dataset):
    def __init__(self, root_dir, df, transform=None):
        self.samples = [str(root_dir) + "/" + str(i) + '.jpg' for i in df['id'].tolist()]
        self.transform = transform

    def __len__(self):  
        return len(self.samples)

    def __getitem__(self, idx):
        img_path = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        else:
            transform_ = transforms.Compose([transforms.ToTensor()])
            image = transform_(image)
        
        return image

In [10]:
tr_images_ds = ImbaDataset(IMAGE_FOLDER, train_df, tr)

In [11]:
batch_size = 32

images = DataLoader(tr_images_ds, batch_size=batch_size, shuffle=False)

In [12]:
sureness50 = []
sureness18 = []
sureness34 = []

for batch in tqdm(images):
    with torch.no_grad():
        image_sure50 = res_net50(batch.to(DEVICE))
        image_sure18 = res_net18(batch.to(DEVICE))
        image_sure34 = res_net34(batch.to(DEVICE))

    sure50 = image_sure50.to('cpu').numpy()
    sure18 = image_sure18.to('cpu').numpy()
    sure34 = image_sure34.to('cpu').numpy()

    sureness50.extend(sure50)
    sureness18.extend(sure18)
    sureness34.extend(sure34)

100%|██████████| 1065/1065 [20:12<00:00,  1.14s/it]


In [36]:
def dif_2max(lst):
    lst1 = [np.exp(-i) for i in lst]
    sm = sum(lst1)
    lst1 = [i / sm for i in lst1]
    l = lst1[::]
    l[l.index(max(l))] = 0
    return float(max(lst1) - max(l))

In [60]:
obj50 = [dif_2max(e) for e in sureness50]
obj18 = [dif_2max(e) for e in sureness18]
obj34 = [dif_2max(e) for e in sureness34]

labels = train_df['label'].to_numpy().tolist()

In [61]:
from sklearn.metrics import roc_auc_score

In [62]:
score50 = roc_auc_score(labels, obj50)
score18 = roc_auc_score(labels, obj18)
score34 = roc_auc_score(labels, obj34)

In [63]:
print(score50, score18, score34)

0.48632752354555875 0.5136873967826682 0.49283674633048224


In [64]:
train_df['obj50'] = obj50
train_df['obj34'] = obj34
train_df['obj18'] = obj18

In [65]:
groups = train_df['card_identifier_id'].unique()

In [66]:
train_lst = train_df.to_numpy().tolist()

In [67]:
d = {i: [[], [], [], []] for i in groups}

for id_, card_identifier_id, label, name, description, obj50, obj18, obj34 in train_lst:
    d[card_identifier_id][0].append(float(label))
    d[card_identifier_id][1].append(float(obj50))
    d[card_identifier_id][2].append(float(obj18))
    d[card_identifier_id][3].append(float(obj34))

In [68]:
for i in groups:
    print(d[i])

[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0], [0.023786332458257675, 0.01853417232632637, 0.00041043758392333984, 0.0037698280066251755, 0.001950044184923172, 0.06799766421318054, 0.061943430453538895, 0.022647196426987648, 0.0054887812584638596, 6.406009197235107e-05, 0.00013692304491996765, 0.003185950219631195], [0.015804961323738098, 0.3167378902435303, 0.008817076683044434, 0.03863991051912308, 0.33697205781936646, 0.016611918807029724, 0.000741083174943924, 0.031112369149923325, 0.051702141761779785, 0.024692194536328316, 0.015058130025863647, 0.025478199124336243], [0.0038588568568229675, 0.03366678208112717, 0.014605900272727013, 0.011620093137025833, 0.15649281442165375, 0.00012528710067272186, 0.018928751349449158, 0.0462457612156868, 0.015426767989993095, 0.0014639422297477722, 0.014087481424212456, 0.004523869603872299]]
[[0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0027109701186418533, 0.014181848615407944, 0.010828429833054543, 0.0708806961774826, 0.

In [69]:
sc50 = 0
sc18 = 0
sc34 = 0

ln = 0

for i in groups:
    if len(set(d[i][0])) == 1:
        # d[i][0] += [0, 1]
        # d[i][1] += [0, 1]
        # d[i][2] += [0, 1]
        # d[i][3] += [0, 1]
        pass
    else:
        s50 = roc_auc_score(d[i][0], d[i][1])
        s18 = roc_auc_score(d[i][0], d[i][2])
        s34 = roc_auc_score(d[i][0], d[i][3])

        sc50 += s50 * len(d[i][0])
        sc18 += s18 * len(d[i][0])
        sc34 += s34 * len(d[i][0])

        ln += len(d[i][0])

        print(s50, sc50, ln)

sc50 /= ln
sc18 /= ln
sc34 /= ln

print(sc50, sc18, sc34)


0.8571428571428572 7.714285714285715 9
0.6545454545454545 18.187012987012988 25
0.4 22.58701298701299 36
0.34065934065934067 29.400199800199804 56
0.7368421052631579 44.137041905462965 76
0.4166666666666667 47.05370857212963 83
0.2 48.45370857212963 90
0.8 54.05370857212963 97
0.6666666666666667 63.387041905462965 111
0.8 72.18704190546296 122
0.13636363636363635 73.95976917819023 135
0.8 79.55976917819022 142
0.7 85.85976917819022 151
0.2857142857142857 88.14548346390451 159
0.5 93.14548346390451 169
0.75 103.64548346390451 183
0.16666666666666666 105.31215013057118 193
0.5918367346938775 113.59786441628546 207
0.33333333333333337 115.93119774961879 214
0.2708333333333333 119.72286441628546 228
0.33333333333333337 122.05619774961879 235
0.2142857142857143 123.98476917819022 244
0.5454545454545454 131.0756782690993 257
0.6000000000000001 136.47567826909932 266
0.4285714285714286 140.33282112624218 275
0.6875 147.20782112624218 285
0.30000000000000004 150.80782112624217 297
0.1999999999